# Chapter 3: LangChain — Orchestrating LLM Applications

### Why This Matters

A single call to an LLM (ask a question, get an answer) only goes so far. Real LLM-powered applications need prompts assembled from templates, structured outputs parsed reliably, multi-step reasoning, and tools the model can call — **LangChain** provides the standard building blocks and vocabulary for composing all of this.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "You are a senior business analyst. Summarize the following customer feedback "
    "in 3 bullet points and suggest one action:\n\n{feedback}"
)

chain = prompt | llm | StrOutputParser()
print(chain.invoke({"feedback": "The app crashes every time I try to checkout..."}))

**The `|` (pipe) syntax is LangChain's signature idea, called the "LangChain Expression Language" (LCEL):** it composes a `prompt`, a `llm`, and an output parser into a single reusable `chain`, where data flows through each stage in order, exactly like piping commands together in a Unix shell. `prompt` fills your template with the actual `{feedback}` text; `llm` sends the resulting prompt to the model; `StrOutputParser()` extracts a clean string from the model's raw response object. `temperature=0` is worth flagging specifically: it makes the model's output as deterministic and repeatable as possible (near-zero randomness in word choice) — the right setting for business-analytical tasks like summarization or classification where consistency matters, as opposed to `temperature` closer to 1 for creative writing tasks where varied phrasing is desirable.

### 3.1 Structured Outputs — Making LLM Responses Machine-Readable

In [ ]:
from pydantic import BaseModel

class FeedbackSummary(BaseModel):     # Pydantic models covered fully in Volume 10
    summary: str
    sentiment: str
    recommended_action: str

# Modern LangChain lets you bind a Pydantic schema directly to the model's output,
# so the LLM's free-text response is parsed into a validated, typed Python object
# rather than a raw string you'd otherwise have to parse yourself.

Feeding raw LLM text output directly into downstream business logic (a database write, an automated alert) is fragile — the model might phrase things slightly differently each time. Binding a **Pydantic schema** (Volume 10) to the model's output forces it to return data in your exact, validated structure, turning an inherently unpredictable text generator into a reliable component of an automated pipeline.

### 3.2 Tools — Letting the Model Take Actions

In [ ]:
# Conceptual sketch: a tool is a typed boundary between the LLM's reasoning and a real action
def calculate_discount(order_total: float, discount_pct: float) -> float:
    """Calculate the discounted order total."""
    return order_total * (1 - discount_pct)

# The LLM decides WHEN to call this tool and with what arguments;
# your application code executes it and returns the result — the model never runs the code itself.

⚠️ **A critical, non-negotiable security principle for any LLM application you build:** the model *decides* which tool to call and with what arguments, but your own application code must *validate those arguments and enforce authorization* before actually executing anything — never treat the LLM's own judgment as a security boundary. An LLM can be manipulated (via "prompt injection" — malicious instructions hidden in user input or retrieved documents) into attempting actions it shouldn't; the actual permission check belongs in your deterministic application code, not in the prompt.

### Cheat Sheet — LangChain

| Concept | Purpose |
|---|---|
| `ChatPromptTemplate` | Reusable prompt with fill-in variables |
| `\|` (LCEL) | Compose prompt → model → parser into one chain |
| `temperature=0` | Deterministic, consistent output (vs. higher = more creative/varied) |
| Structured output (Pydantic) | Force the model's response into a validated schema |
| Tools | Typed functions the model can request; your code validates & executes |

---